In [ ]:
from google.colab import files
uploaded = files.upload()

Saving combined_news.csv to combined_news.csv


In [ ]:
"""
TrustLens - News Articles Baseline Model

"""

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import joblib
import json
from datetime import datetime

# Configuration
DATASET_PATH = 'combined_news.csv'
TEST_SIZE = 0.2
RANDOM_STATE = 42

# =============================================================================
# 1. Load Data
# =============================================================================


df = pd.read_csv(DATASET_PATH)
print(f"Dataset loaded: {len(df)} samples")


# =============================================================================
# 2. Data Preparation
# =============================================================================



# Clean data (using correct column names)
df = df.dropna(subset=['title', 'Article', 'MachineGen'])

# Combine title and Article
df['text'] = df['title'].astype(str) + '. ' + df['Article'].astype(str)

# Create labels
df['label'] = df['MachineGen'].astype(int)

# Remove duplicates
df = df.drop_duplicates(subset=['text'])


# =============================================================================
# 3. Balance the Dataset
# =============================================================================


df_human = df[df['label'] == 0]
df_ai = df[df['label'] == 1]

# Take minimum count from both classes
min_count = min(len(df_human), len(df_ai))

df_human_balanced = df_human.sample(n=min_count, random_state=RANDOM_STATE)
df_ai_balanced = df_ai.sample(n=min_count, random_state=RANDOM_STATE)

df_balanced = pd.concat([df_human_balanced, df_ai_balanced]).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

print(f"Balanced dataset: {len(df_balanced)} samples")
print(f"Label distribution:\n{df_balanced['label'].value_counts()}")

# =============================================================================
# 4. Train-Test Split
# =============================================================================



X = df_balanced['text']
y = df_balanced['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

print(f"Training samples: {len(X_train)}")
print(f"Testing samples: {len(X_test)}")

# =============================================================================
# 5. TF-IDF Vectorization
# =============================================================================


vectorizer = TfidfVectorizer(
    max_features=5000, # يحدد أهم 5000 كلمة
    ngram_range=(1, 2),# يأخذ كلمة واحدة أو كلمتين متصلتين
    min_df=2,# يحذف الكلمات اللي تظهر مره وحده
    max_df=0.8 #يحذف الكلمات المكررة80٪ من المقالات
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)



# =============================================================================
# 6. Model Training
# =============================================================================



model = LogisticRegression(
    max_iter=1000,
    random_state=RANDOM_STATE,
    class_weight='balanced' #يوازن البيانات
)

model.fit(X_train_tfidf, y_train)

print("Training completed")

# =============================================================================
# 7. Evaluation
# =============================================================================


y_pred = model.predict(X_test_tfidf)
accuracy = accuracy_score(y_test, y_pred)

print(f"\nAccuracy: {accuracy*100:.2f}%")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Human', 'AI']))

print("\nConfusion Matrix:")
cm = confusion_matrix(y_test, y_pred)
print(f"              Predicted")
print(f"           Human    AI")
print(f"Human      {cm[0][0]:5d}  {cm[0][1]:5d}")
print(f"AI         {cm[1][0]:5d}  {cm[1][1]:5d}")

# =============================================================================
# 8. Save Model
# =============================================================================

print("\nSaving model...")


pipeline = Pipeline([
    ('tfidf', vectorizer),
    ('classifier', model)
])

# احفظ الـ Pipeline كامل
joblib.dump(pipeline, 'news_baseline_pipeline.pkl')

metadata = {
    'model': 'Logistic Regression',
    'dataset_size': len(df_balanced),
    'train_size': len(X_train),
    'test_size': len(X_test),
    'accuracy': float(accuracy),
    'vocabulary_size': len(vectorizer.vocabulary_),
    'date': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
}

with open('model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=4)

print("Model saved successfully")

# =============================================================================
# 9. Chunk Function
# =============================================================================

def split_into_3_chunks(text):
    words = text.split()
    if len(words) <= 3:
        return [text]

    chunk_size = len(words) // 3
    chunks = [
        " ".join(words[:chunk_size]),
        " ".join(words[chunk_size:2*chunk_size]),
        " ".join(words[2*chunk_size:])
    ]
    return chunks

# =============================================================================
# 10. Test Function
# =============================================================================

def predict_article(title, content):
    """
    تقسم المقالة لـ 3 chunks وتحسب سكور كل chunk
    بعدين تطلع متوسط السكورات للمقالة كلها
    """
    loaded_pipeline = joblib.load('news_baseline_pipeline.pkl')

    text = f"{title}. {content}"
    chunks = split_into_3_chunks(text)

    human_scores = []
    ai_scores = []

    # التنبؤ على كل chunk
    for chunk in chunks:
        probabilities = loaded_pipeline.predict_proba([chunk])[0]
        human_scores.append(probabilities[0])
        ai_scores.append(probabilities[1])

    # متوسط السكورات كل الـ chunks
    final_human = sum(human_scores) / len(human_scores)
    final_ai = sum(ai_scores) / len(ai_scores)

    return {
        'prediction': 'AI' if final_ai > final_human else 'Human',
        'confidence': round(max(final_human, final_ai) * 100, 2),
        'probabilities': {
            'human': round(final_human * 100, 2),
            'ai': round(final_ai * 100, 2)
        },
        'chunks_detail': [
            {
                'chunk_id': i + 1,
                'text': chunk[:80] + '...',
                'human': round(h * 100, 2),
                'ai': round(a * 100, 2)
            }
            for i, (chunk, h, a) in enumerate(zip(chunks, human_scores, ai_scores))
        ]
    }

# =============================================================================
# 11. Test Examples
# =============================================================================



test_cases = [
    {
        'title': 'Pakistan stocks end record high',
        'content': 'KARACHI: The stock market ended on a record high today after investors showed confidence in the economic reforms announced by the government last week.'
    },
    {
        'title': 'Artificial Intelligence Advances in Healthcare',
        'content': 'According to recent studies, artificial intelligence systems have demonstrated significant improvements in diagnosing diseases at an early stage, with an accuracy rate exceeding 95 percent.'
    }
]

for i, case in enumerate(test_cases, 1):
    result = predict_article(case['title'], case['content'])
    print(f"\nTest {i}:")
    print(f"  Title: {case['title']}")
    print(f"  Prediction: {result['prediction']}")
    print(f"  Confidence: {result['confidence']}%")
    print(f"  Human: {result['probabilities']['human']}% | AI: {result['probabilities']['ai']}%")
    print(f"  Chunks Detail:")
    for chunk in result['chunks_detail']:
        print(f"    Chunk {chunk['chunk_id']}: Human {chunk['human']}% | AI {chunk['ai']}% | {chunk['text']}")

print("\n" + "="*60)
print("Baseline Model Training Complete")
print("="*60)

Dataset loaded: 3692 samples
Balanced dataset: 2000 samples
Label distribution:
label
1    1000
0    1000
Name: count, dtype: int64
Training samples: 1600
Testing samples: 400
Training completed

Accuracy: 95.25%

Classification Report:
              precision    recall  f1-score   support

       Human       0.94      0.96      0.95       200
          AI       0.96      0.94      0.95       200

    accuracy                           0.95       400
   macro avg       0.95      0.95      0.95       400
weighted avg       0.95      0.95      0.95       400


Confusion Matrix:
              Predicted
           Human    AI
Human        193      7
AI            12    188

Saving model...
Model saved successfully

Test 1:
  Title: Pakistan stocks end record high
  Prediction: Human
  Confidence: 50.76%
  Human: 50.76% | AI: 49.24%
  Chunks Detail:
    Chunk 1: Human 76.6% | AI 23.4% | Pakistan stocks end record high. KARACHI: The stock market...
    Chunk 2: Human 35.91% | AI 64.09% | end

# New Section